In [ ]:
pip install -r requirements-benchmarks.txt

# Генератор данных (data_generator.py)

Эта ячейка создаст два больших Parquet-файла, которые мы будем использовать во всех тестах. На генерацию может уйти несколько минут.

In [2]:
# data_generator.py
import pandas as pd
import numpy as np
import os

NUM_TRANSACTIONS = 100_000_000
NUM_USERS = 5_000_000
NUM_CATEGORIES = 100
DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)

transactions = pd.DataFrame({
    'user_id': np.random.randint(1, NUM_USERS + 1, size=NUM_TRANSACTIONS),
    'category_id': np.random.randint(1, NUM_CATEGORIES + 1, size=NUM_TRANSACTIONS, dtype=np.int16),
    'timestamp': pd.to_datetime(np.random.randint(
        pd.to_datetime('2020-03-01').value, 
        pd.to_datetime('2026-03-01').value, 
        size=NUM_TRANSACTIONS
    ), unit='ns'),
    'revenue': np.random.uniform(100, 5000, size=NUM_TRANSACTIONS).astype(np.float32)
})
transactions_path = os.path.join(DATA_DIR, 'transactions.parquet')
transactions.to_parquet(transactions_path, index=False)

users = pd.DataFrame({
    'user_id': np.arange(1, NUM_USERS + 1),
    'registration_date': pd.to_datetime(np.random.randint(
        pd.to_datetime('2020-03-01').value, 
        pd.to_datetime('2026-03-01').value, 
        size=NUM_USERS
    ), unit='ns')
})
users_path = os.path.join(DATA_DIR, 'users.parquet')
users.to_parquet(users_path, index=False)

# Бенчмарки и анти-паттерны

Для каждого бенчмарка ниже будет по 2 ячейки: одна для pandas, другая для fireducks

Перед запуском этих ячеек установите memory-profiler и его расширение в Jupyter:

In [3]:
%load_ext memory_profiler

## Бенчмарк: GroupBy + Agg

Классическая аналитическая задача. Здесь FireDucks должен показать себя во всей красе.

### Pandas версия:

In [4]:
%%time
from memory_profiler import memory_usage
import pandas as pd

def run_pandas_groupby():
    df = pd.read_parquet('data/transactions.parquet')
    result = df.groupby(['category_id']).agg(
        total_revenue=('revenue', 'sum'),
        avg_revenue=('revenue', 'mean'),
        transaction_count=('user_id', 'count')
    )
    print(result.head())

mem_usage = memory_usage(run_pandas_groupby, interval=0.1)
print(f"Pandas Peak Memory: {max(mem_usage):.2f} MiB")

             total_revenue  avg_revenue  transaction_count
category_id                                               
1             2.550682e+09  2549.366455            1000516
2             2.548558e+09  2551.167725             998977
3             2.551114e+09  2551.708740             999767
4             2.552691e+09  2551.721680            1000380
5             2.551984e+09  2550.984131            1000392
Pandas Peak Memory: 8061.43 MiB
CPU times: user 5.54 s, sys: 5.42 s, total: 11 s
Wall time: 6.38 s


### FireDucks версия:

In [5]:
%%time
from memory_profiler import memory_usage
import fireducks.pandas as pd

def run_fireducks_groupby():
    df = pd.read_parquet('data/transactions.parquet')
    result = df.groupby(['category_id']).agg(
        total_revenue=('revenue', 'sum'),
        avg_revenue=('revenue', 'mean'),
        transaction_count=('user_id', 'count')
    )
    print(result.head())

mem_usage = memory_usage(run_fireducks_groupby, interval=0.1)
print(f"FireDucks Peak Memory: {max(mem_usage):.2f} MiB")

             total_revenue  avg_revenue  transaction_count
category_id                                               
1             2.550682e+09  2549.366342            1000516
2             2.548558e+09  2551.167691             998977
3             2.551114e+09  2551.708841             999767
4             2.552692e+09  2551.721889            1000380
5             2.551984e+09  2550.984066            1000392
FireDucks Peak Memory: 7918.26 MiB
CPU times: user 5.92 s, sys: 8.96 s, total: 14.9 s
Wall time: 1.96 s


## Бенчмарк: Merge (Join)

Объединение двух больших DataFrame — еще одна задача, где параллелизм решает.

### Pandas версия:

In [6]:
%%time
from memory_profiler import memory_usage
import pandas as pd

def run_pandas_merge():
    transactions = pd.read_parquet('data/transactions.parquet')
    users = pd.read_parquet('data/users.parquet')
    merged_df = pd.merge(transactions, users, on='user_id', how='inner')
    print(merged_df.shape)

mem_usage = memory_usage(run_pandas_merge, interval=0.1)
print(f"Pandas Peak Memory: {max(mem_usage):.2f} MiB")

(100000000, 5)
Pandas Peak Memory: 17220.05 MiB
CPU times: user 36.5 s, sys: 12.4 s, total: 48.8 s
Wall time: 43.1 s


### FireDucks версия:

In [7]:
%%time
from memory_profiler import memory_usage
import fireducks.pandas as pd
import pyarrow as pa

def run_fireducks_merge():
    transactions = pd.read_parquet('data/transactions.parquet')
    users = pd.read_parquet('data/users.parquet')
    merged_df = pd.merge(transactions, users, on='user_id', how='inner')
    print(merged_df.shape)
    
mem_usage = memory_usage(run_fireducks_merge, interval=0.1)
print(f"FireDucks Peak Memory: {max(mem_usage):.2f} MiB")

(100000000, 5)
FireDucks Peak Memory: 15703.75 MiB
CPU times: user 46.2 s, sys: 17 s, total: 1min 3s
Wall time: 6.19 s


## Бенчмарк: Full ETL Chain (Главный тест)

Здесь оптимизатор запросов FireDucks должен показать максимальную эффективность, объединив несколько шагов в один.

### Pandas версия:

In [8]:
%%time
from memory_profiler import memory_usage
import pandas as pd

def run_pandas_etl():
    transactions = pd.read_parquet('data/transactions.parquet')
    users = pd.read_parquet('data/users.parquet')
    
    # Фильтрация
    filtered_transactions = transactions[transactions['timestamp'] > '2022-01-01']
    
    # Объединение
    merged_df = pd.merge(filtered_transactions, users, on='user_id', how='inner')
    
    # Агрегация
    report = merged_df.groupby(merged_df['registration_date'].dt.year).agg(
        total_revenue=('revenue', 'sum')
    )
    print(report)

mem_usage = memory_usage(run_pandas_etl, interval=0.1)
print(f"Pandas Peak Memory: {max(mem_usage):.2f} MiB")

                   total_revenue
registration_date               
2020                2.474168e+10
2021                2.949560e+10
2022                2.943396e+10
2023                2.948432e+10
2024                2.952993e+10
2025                2.946203e+10
2026                4.761278e+09
Pandas Peak Memory: 24484.70 MiB
CPU times: user 32.4 s, sys: 13.3 s, total: 45.7 s
Wall time: 39.7 s


### FireDucks версия:

In [9]:
%%time
from memory_profiler import memory_usage
import fireducks.pandas as pd

def run_fireducks_etl():
    transactions = pd.read_parquet('data/transactions.parquet')
    users = pd.read_parquet('data/users.parquet')
    
    # Фильтрация
    filtered_transactions = transactions[transactions['timestamp'] > '2022-01-01']
    
    # Объединение
    merged_df = pd.merge(filtered_transactions, users, on='user_id', how='inner')
    
    # Агрегация
    report = merged_df.groupby(merged_df['registration_date'].dt.year).agg(
        total_revenue=('revenue', 'sum')
    )
    print(report)

mem_usage = memory_usage(run_fireducks_etl, interval=0.1)
print(f"FireDucks Peak Memory: {max(mem_usage):.2f} MiB")

                   total_revenue
registration_date               
2020                2.474166e+10
2021                2.949558e+10
2022                2.943390e+10
2023                2.948430e+10
2024                2.952989e+10
2025                2.946202e+10
2026                4.761277e+09
FireDucks Peak Memory: 21540.16 MiB
CPU times: user 40.5 s, sys: 33.7 s, total: 1min 14s
Wall time: 7.18 s


### Анти-паттерн: Complex .apply()

Здесь мы ожидаем, что FireDucks не даст ускорения. 

#### Внимание: для этого теста лучше взять срез данных (например, 10 млн строк), иначе на .apply можно ждать очень долго.

### Pandas версия:

In [10]:
%%time
from memory_profiler import memory_usage
import pandas as pd

# Берем срез данных, чтобы тест завершился за разумное время
df_sample = pd.read_parquet('data/transactions.parquet').head(10_000_000)

def complex_logic(row):
    if row['revenue'] > 4000 and row['category_id'] in [10, 20, 30]:
        return 'Premium High-Value'
    elif row['revenue'] < 500:
        return 'Low-Value'
    else:
        return 'Standard'

def run_pandas_apply():
    df_sample['segment'] = df_sample.apply(complex_logic, axis=1)
    print(df_sample['segment'].value_counts())

mem_usage = memory_usage(run_pandas_apply, interval=0.1)
print(f"Pandas Peak Memory: {max(mem_usage):.2f} MiB")

segment
Standard              9121755
Low-Value              816845
Premium High-Value      61400
Name: count, dtype: int64
Pandas Peak Memory: 27326.47 MiB
CPU times: user 1min 19s, sys: 9.91 s, total: 1min 29s
Wall time: 1min 25s


### FireDucks версия:

In [11]:
%%time
from memory_profiler import memory_usage
import fireducks.pandas as pd
import pandas as pd_orig # для чтения

# здесь хватит и 1кк записей, чтобы показать неэффективность, чтобы не ждать вечность
df_sample_orig = pd_orig.read_parquet('data/transactions.parquet').head(1_000_000)

# Конвертируем в FireDucks DataFrame
df_sample = pd.DataFrame(df_sample_orig)

def complex_logic(row):
    if row['revenue'] > 4000 and row['category_id'] in [10, 20, 30]:
        return 'Premium High-Value'
    elif row['revenue'] < 500:
        return 'Low-Value'
    else:
        return 'Standard'

def run_fireducks_apply():
    # FireDucks откатится к режиму pandas для этой операции
    df_sample['segment'] = df_sample.apply(complex_logic, axis=1)
    print(df_sample['segment'].value_counts())

mem_usage = memory_usage(run_fireducks_apply, interval=0.1)
print(f"FireDucks Peak Memory: {max(mem_usage):.2f} MiB")

segment
Standard              912430
Low-Value              81417
Premium High-Value      6153
Name: count, dtype: int64
FireDucks Peak Memory: 31519.81 MiB
CPU times: user 6min 35s, sys: 21.9 s, total: 6min 57s
Wall time: 7min 17s


### Причина:

Для выполнения Python-функции на каждой строке, ```FireDucks``` вынужден для каждой из 10 миллионов строк конвертировать данные из своего внутреннего высокопроизводительного формата в стандартный объект ```pandas.Series```. Эти накладные расходы на конвертацию делают процесс намного медленнее, чем нативный, хоть и не быстрый, цикл в ```pandas```.

### Вывод: 
Использование ```.apply(axis=1)``` полностью нивелирует все преимущества ```FireDucks``` и является главным анти-паттерном, которого следует избегать.